# Real-Time Autoregressive Generation with Local Ollama

This interactive notebook connects to your local Ollama instance (running `llama3.2`). 
Every time you click **Generate 1 Token**, it sends the *entire current context* to the API, restricts the output to exactly `max_tokens=1`, and visually appends that new token to the context window.

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import requests
from openai import OpenAI
import json

# --- API Configuration ---
OLLAMA_ENDPOINT = "http://localhost:11434/v1/completions"
MODEL_NAME = "llama3"
MODEL_URL = "http://localhost:11434/v1"

# --- UI Components ---
prompt_input = widgets.Textarea(
    value="The robot learned to",
    description='Prompt:',
    layout=widgets.Layout(width='80%', height='60px')
)

btn_gen = widgets.Button(description="Generate 1 Token", button_style='primary')
btn_reset = widgets.Button(description="Reset Context", button_style='warning')
buttons = widgets.HBox([btn_gen, btn_reset])
output_area = widgets.Output()

# --- State ---
global current_context
current_context = prompt_input.value

def call_ollama(prompt_text):
    headers = {"Content-Type": "application/json"}
    data = {
        "model": MODEL_NAME,
        "prompt": prompt_text,
        "max_completion_tokens": 2,
        "max_tokens": 2,       # Forces exactly 1 token
        "temperature": 0.0,    # Deterministic generation
        # "stop": ["<|eot_id|>", "<|end_of_text|>"]
    }
    try:
        response = requests.post(
            OLLAMA_ENDPOINT,
            headers=headers,
            json=data)
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["text"]
    except requests.exceptions.ConnectionError:
        return " [ERROR: Could not connect to localhost:11434]"
    except Exception as e:
        return f" [ERROR: {str(e)}]"

def update_display(highlight_token=""):
    with output_area:
        clear_output(wait=True)
        # Separate the base text from the newest token for highlighting
        global current_context
        base_text = current_context[:-len(highlight_token)] if highlight_token else current_context
        
        html = f"""
        <div style="padding:15px; border:2px dashed #90caf9; border-radius:8px; font-family:monospace; font-size:16px; background:#f8f9fa; white-space:pre-wrap; margin-top:15px;">
            {base_text}<span style="background-color:#d4edda; border:1px solid #28a745; border-radius:3px; font-weight:bold; padding:0 2px;">{highlight_token}</span>
        </div>
        """
        display(HTML(html))

def on_generate(b):
    global current_context
    btn_gen.disabled = True
    
    with output_area:
        clear_output(wait=True)
        display(HTML("<i style='color:gray;'>Calling Ollama API... predicting next token...</i>"))
        
    next_token = call_ollama(current_context)
    current_context += ' '+ next_token
    
    update_display(highlight_token=next_token)
    prompt_input.value = current_context
    btn_gen.disabled = False

def on_reset(b):
    global current_context
    current_context = prompt_input.value
    update_display()

def on_prompt_change(change):
    on_reset(None)

# --- Bind Events ---
btn_gen.on_click(on_generate)
btn_reset.on_click(on_reset)
prompt_input.observe(on_prompt_change, names='value')

# --- Initial Display ---
display(widgets.VBox([prompt_input, buttons, output_area]))
update_display()
